# 🎬 Video Caption Generator - Google Colab

Comprehensive video captioning with LLaVA JoyCaption using:
- ⏱️ Time-based frame sampling (every 2 seconds)
- 📝 Individual frame captioning with timestamp→caption dict structure
- 🎯 Full video coverage without limits
- 📊 Structured JSON output with direct timestamp lookup

## 🚀 Quick Start
1. **Runtime** → **Change runtime type** → Select **GPU** (T4 recommended)
2. **Runtime** → **Run all** (runs all cells in order)
3. **Edit R2 credentials** in cell 13 with your actual credentials
4. **Edit JOBS list** in cell 15 to add your videos
5. Wait for processing to complete
6. View JSON results at the bottom

## ⚠️ Important
**You must run cells 1-16 in order before running the processing cell (18)**

If you get a `NameError`, it means you skipped cells. Solution:
- Go to: **Runtime** → **Run all**
- Or manually run cells 1-16 sequentially

## 📋 Cell Structure
- **Cells 1-6**: Setup, environment, model loading
- **Cells 7-12**: Helper functions, frame extraction, captioning
- **Cell 13**: R2 cloud storage configuration
- **Cell 15**: JOBS list (add your videos here)
- **Cell 16**: Main processing function
- **Cell 18**: Process all jobs (run this to start)
- **Cell 19**: Display results

## 📊 Output Structure
```json
{
  "id": "scene_000.mp4",
  "status": "completed",
  "captions": {
    "0:00": "Person enters room wearing blue jacket",
    "0:02": "Person walks toward table",
    "0:04": "Person picks up red book from table"
  },
  "metadata": {
    "duration_seconds": 120.5,
    "frames_sampled": 60,
    "sampling_interval": 2.0,
    "processing_time_seconds": 45.2
  }
}
```

## ⚙️ Setup & Installation

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies
!pip install -q "transformers>=4.44.0" accelerate bitsandbytes boto3 pillow

# Install ffmpeg for video processing
!apt-get -qq install ffmpeg > /dev/null 2>&1

print("✅ All dependencies installed")

In [ ]:
# Environment configuration
import os

# Avoid torchvision in transformers pipeline
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

# Reduce CUDA fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# Disable fused SDPA to avoid dtype conflicts
import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

print("✅ Environment configured")

## 🤖 Model Loading

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image

MODEL_NAME = "fancyfeast/llama-joycaption-alpha-two-hf-llava"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"📥 Loading model: {MODEL_NAME}")
print("⏳ This may take 2-3 minutes...\n")

# Load processor
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("✅ Processor loaded")

# Load model in float16 (no quantization)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # float16 without quantization
    device_map="auto",
    attn_implementation="eager",
    trust_remote_code=True,
)
model.eval()

print(f"✅ Model loaded on {DEVICE}")
print(f"⚠️  Note: Using float16 without quantization (~16GB VRAM)")

## 🛠️ Helper Functions

In [ ]:
import subprocess
import tempfile
from pathlib import Path
from typing import List, Tuple

# Configuration
SAMPLING_INTERVAL = 2.0  # Extract frame every 2 seconds
FRAMES_PER_BATCH = 5     # Process 5 frames together (~10 seconds)
MAX_SIDE = 672           # Max image dimension for VRAM efficiency


def _normalize_inputs_for_generate(inputs, device):
    """FIXED: Properly normalize dtypes for model.generate()
    
    Critical fixes:
    1. Force attention_mask to int64 regardless of input dtype
    2. Move to device AFTER dtype conversion (not before)
    3. Ensure pixel_values are float16
    """
    fixed = {}
    
    for k, v in inputs.items():
        if not hasattr(v, "to"):
            fixed[k] = v
            continue
            
        # Step 1: Fix dtype BEFORE moving to device
        if k == "pixel_values":
            if v.dtype != torch.float16:
                v = v.to(torch.float16)
        elif k == "input_ids":
            if v.dtype != torch.int64:
                v = v.to(torch.int64)
        elif k in ("attention_mask", "pixel_attention_mask", "cross_attention_mask"):
            # CRITICAL FIX: Always force int64 for mask tensors
            if v.dtype != torch.int64:
                v = v.to(torch.int64)
        
        # Step 2: Now move to device
        v = v.to(device)
        fixed[k] = v

    return fixed


def probe_duration(video_path: str) -> float:
    """Get video duration in seconds"""
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        video_path
    ]
    output = subprocess.check_output(cmd, stderr=subprocess.STDOUT).decode().strip()
    return float(output)


def extract_frame_at_time(video_path: str, timestamp: float, output_path: str):
    """Extract single frame at specific timestamp"""
    cmd = [
        "ffmpeg",
        "-ss", f"{timestamp:.3f}",
        "-i", video_path,
        "-vframes", "1",
        "-y",
        output_path
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def downscale_image(img: Image.Image, max_side: int = MAX_SIDE) -> Image.Image:
    """Downscale image for VRAM efficiency"""
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    scale = max_side / float(max(w, h))
    return img.resize((int(w*scale), int(h*scale)), Image.BICUBIC)


def format_time(seconds: float) -> str:
    """Format seconds as M:SS"""
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins}:{secs:02d}"


print("✅ Helper functions loaded")

## 🎥 Frame Extraction & Batching

In [ ]:
def extract_frames_at_intervals(video_bytes: bytes, interval: float = SAMPLING_INTERVAL) -> Tuple[List[Image.Image], List[float]]:
    """Extract frames at regular time intervals
    
    Returns:
        Tuple of (frames, timestamps)
    """
    frames = []
    timestamps = []
    
    with tempfile.TemporaryDirectory() as td:
        # Write video to temp file
        video_path = Path(td) / "video.mp4"
        video_path.write_bytes(video_bytes)
        
        # Get duration
        duration = probe_duration(str(video_path))
        if duration <= 0:
            raise ValueError("Could not determine video duration")
        
        # Calculate timestamps
        current_time = 0.0
        while current_time < duration:
            timestamps.append(current_time)
            current_time += interval
        
        # Add final frame if needed
        if timestamps[-1] < duration - 0.5:
            timestamps.append(duration - 0.5)
        
        print(f"📊 Video duration: {duration:.1f}s, sampling {len(timestamps)} frames")
        
        # Extract frames
        for i, ts in enumerate(timestamps):
            frame_path = Path(td) / f"frame_{i:04d}.jpg"
            extract_frame_at_time(str(video_path), ts, str(frame_path))
            
            img = Image.open(frame_path).convert("RGB")
            img = downscale_image(img)
            frames.append(img)
    
    return frames, timestamps


def batch_frames(frames: List[Image.Image], timestamps: List[float], batch_size: int = FRAMES_PER_BATCH) -> List[dict]:
    """Group frames into batches for unified captioning"""
    batches = []
    
    for i in range(0, len(frames), batch_size):
        batch_frames = frames[i:i + batch_size]
        batch_timestamps = timestamps[i:i + batch_size]
        
        start_time = batch_timestamps[0]
        end_time = batch_timestamps[-1] + SAMPLING_INTERVAL
        
        batches.append({
            "batch_id": len(batches) + 1,
            "frames": batch_frames,
            "timestamps": batch_timestamps,
            "start_time": start_time,
            "end_time": end_time,
        })
    
    return batches


print("✅ Frame extraction functions loaded")

## 💬 Captioning Functions

In [ ]:
def caption_single_image(img: Image.Image, prompt: str) -> str:
    """Generate caption for a single image (proven working implementation)"""
    convo = [
        {"role": "system", "content": "You are a concise, visual captioner."},
        {"role": "user", "content": prompt}
    ]
    tmpl = processor.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)

    raw_inputs = processor(text=[tmpl], images=[img], return_tensors="pt", padding=True)
    inputs = _normalize_inputs_for_generate(raw_inputs, DEVICE)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            use_cache=True
        )[0]
    
    out = out[inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()


def caption_batch(images: List[Image.Image], prompt: str = None) -> List[str]:
    """Generate individual captions for each frame

    Returns individual captions for each frame (no merging).
    This approach:
    - Works with the model's architecture (single-image only)
    - Uses less VRAM (~5-8GB per frame vs ~20GB for batch)
    - Clears VRAM between frames to prevent OOM errors
    - Returns individual captions for timestamp→caption dict structure
    """
    if prompt is None:
        prompt = "Write a concise, descriptive caption for this video frame."

    captions = []
    total_frames = len(images)

    for i, img in enumerate(images, 1):
        # Process one frame at a time
        cap = caption_single_image(img, prompt)
        captions.append(cap)

        # Critical: Clear VRAM after each frame to prevent OOM
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return captions


print("✅ Captioning functions loaded (returns individual captions)")

In [ ]:
import boto3

# R2 Configuration - Credentials loaded from Untitled0.ipynb
R2_ACCOUNT_ID = "041c5851f7d1adb7675ea82a59f7cbe4"
R2_ACCESS_KEY_ID = "fb87a0febeaa764fe2a0ef1eb956a74e"
R2_SECRET_KEY = "eabd1a2a5ee98be407d165352486f548b61526492a1daf09a605ae29d8e75a26"
R2_BUCKET = "storygen"

# Initialize S3 client for R2
s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_KEY,
    region_name="auto"
)

print("✅ R2 client configured")
print(f"   Bucket: {R2_BUCKET}")
print(f"   Endpoint: https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com")

In [ ]:
## 📋 Jobs Configuration

# Define your videos to process

In [ ]:
# Jobs list - add your videos here
JOBS = [
    {
        "id": "scene_000.mp4",
        "clip_path": "assets/clips/scene_000.mp4",
        "status": "pending",
        "caption": None
    },
    # Add more jobs here as needed
    # {
    #     "id": "scene_001.mp4",
    #     "clip_path": "assets/clips/scene_001.mp4",
    #     "status": "pending",
    #     "caption": None
    # },
]

print(f"✅ Jobs configured: {len(JOBS)} videos to process")
print("\n📋 Jobs list:")
for i, job in enumerate(JOBS, 1):
    status_icon = "⏳" if job.get("status") in (None, "pending") else "✅"
    print(f"   {status_icon} {i}. {job.get('clip_path', job.get('id'))}")

In [ ]:
import json
import time
from IPython.display import display, HTML


def process_one_job(job: dict, interval: float = SAMPLING_INTERVAL, batch_size: int = FRAMES_PER_BATCH) -> dict:
    """Process a single video job with time-based sampling and individual frame captioning
    
    Args:
        job: Job dict with 'clip_path' key
        interval: Frame sampling interval in seconds
        batch_size: Number of frames per batch
        
    Returns:
        Updated job dict with timestamp→caption dict structure
    """
    clip_path = job.get("clip_path", job.get("id"))
    
    print(f"\n{'='*80}")
    print(f"🎬 Processing: {clip_path}")
    print(f"{'='*80}\n")
    
    start_time = time.time()
    
    try:
        # Download from R2
        print(f"☁️  Downloading from R2...")
        obj = s3.get_object(Bucket=R2_BUCKET, Key=clip_path)
        video_bytes = obj["Body"].read()
        print(f"✅ Downloaded {len(video_bytes)/1024/1024:.1f} MB\n")
        
        # Extract frames
        print("🔍 Extracting frames...")
        frames, timestamps = extract_frames_at_intervals(video_bytes, interval)
        
        # Create batches
        batches = batch_frames(frames, timestamps, batch_size)
        print(f"📦 Created {len(batches)} batches ({batch_size} frames each)\n")
        
        # Build timestamp→caption dict
        captions_dict = {}
        
        for batch in batches:
            print(f"   🔄 Processing batch {batch['batch_id']}/{len(batches)}... ", end="", flush=True)
            
            # Get individual captions for each frame in batch
            individual_captions = caption_batch(batch["frames"])
            
            # Map each timestamp to its caption
            for timestamp, caption in zip(batch["timestamps"], individual_captions):
                time_key = format_time(timestamp)
                captions_dict[time_key] = caption
            
            print(f"✅")
            # Show first caption as sample
            sample_caption = individual_captions[0]
            print(f"      📝 Sample: {sample_caption[:70]}{'...' if len(sample_caption) > 70 else ''}\n")
        
        # Calculate metadata
        processing_time = time.time() - start_time
        duration = timestamps[-1] + interval
        
        # Update job with new structure
        job["status"] = "completed"
        job["captions"] = captions_dict
        job["metadata"] = {
            "duration_seconds": round(duration, 1),
            "frames_sampled": len(frames),
            "sampling_interval": interval,
            "processing_time_seconds": round(processing_time, 2)
        }
        
        print(f"{'='*80}")
        print(f"✅ Completed in {processing_time:.1f}s")
        print(f"   📊 {len(captions_dict)} frame captions | {duration:.1f}s video | {len(frames)} frames")
        print(f"{'='*80}")
        
    except Exception as e:
        job["status"] = "failed"
        job["error"] = str(e)
        print(f"\n{'='*80}")
        print(f"❌ Failed: {e}")
        print(f"{'='*80}")
    
    return job


print("✅ Job processing function loaded (timestamp→caption dict structure)")

## 🚀 Process All Jobs

Run this cell to process all pending jobs

In [ ]:
# Check if all required functions and variables are defined
try:
    process_one_job
    s3
    R2_BUCKET
    JOBS
except NameError as e:
    print("=" * 80)
    print("❌ ERROR: Required dependencies not loaded!")
    print("=" * 80)
    print(f"\n⚠️  Missing: {e}")
    print("\n📋 Solution: Run cells in order:")
    print("   1. Go to: Runtime → Run all")
    print("   2. Or manually run cells 1-16 before running this cell")
    print("\n💡 Cells needed:")
    print("   - Cells 1-12: Setup, model, functions")
    print("   - Cell 13: R2 configuration")
    print("   - Cell 15: JOBS list")
    print("   - Cell 16: process_one_job function")
    print("=" * 80)
    raise

# Validate R2 connection before processing
print("🔍 Validating R2 connection...")
try:
    test_response = s3.list_objects_v2(Bucket=R2_BUCKET, MaxKeys=1)
    print(f"✅ R2 connection successful")
    print(f"   Bucket: {R2_BUCKET}")
except Exception as e:
    print("=" * 80)
    print("❌ ERROR: Failed to connect to R2")
    print("=" * 80)
    print(f"\n⚠️  Error: {e}")
    print("\n📋 Common issues:")
    print("   - Invalid credentials (check cell 13)")
    print("   - Wrong bucket name")
    print("   - Network connectivity issues")
    print("=" * 80)
    raise

# Process all pending jobs
pending_jobs = [j for j in JOBS if j.get("status") in (None, "pending")]

if not pending_jobs:
    print("🎉 No pending jobs to process!")
else:
    print(f"\n📋 Processing {len(pending_jobs)} pending jobs...\n")
    
    for i, job in enumerate(pending_jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# JOB {i}/{len(pending_jobs)}")
        print(f"{'#'*80}")
        
        process_one_job(job, interval=SAMPLING_INTERVAL, batch_size=FRAMES_PER_BATCH)
    
    # Summary
    print(f"\n\n{'='*80}")
    print(f"📊 PROCESSING SUMMARY")
    print(f"{'='*80}\n")
    
    completed = [j for j in JOBS if j.get("status") == "completed"]
    failed = [j for j in JOBS if j.get("status") == "failed"]
    
    print(f"✅ Completed: {len(completed)}/{len(pending_jobs)}")
    print(f"❌ Failed: {len(failed)}/{len(pending_jobs)}")
    
    if failed:
        print(f"\n⚠️  Failed jobs:")
        for job in failed:
            print(f"   - {job.get('clip_path')}: {job.get('error')}")
    
    print(f"\n{'='*80}\n")